# Export embeddings and land cover

Run `starter.ipynb` first to save `outputs/model.pt`. This notebook writes the middle-layer embeddings as float32 and class predictions as uint8 COGs on the source grid. Ocean and excluded labels are masked; lakes are included.

In [1]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import rasterio
import torch
from rasterio.shutil import copy as raster_copy
from rasterio.windows import Window
from torch import nn


class Wrap(nn.Module):
    def forward(self, coords):
        coords.deg2rad_()
        features = coords.repeat_interleave(2, dim=-1)
        features[:, 0::2].cos_()
        features[:, 1::2].sin_()
        return features


def load_model(checkpoint, device):
    saved = torch.load(checkpoint, map_location=device, weights_only=True)
    encoder = nn.Sequential(
        Wrap(),
        nn.Linear(4, saved["input_dim"]),
        nn.ReLU(inplace=True),
        nn.Linear(saved["input_dim"], saved["hidden_dim"]),
        nn.ReLU(inplace=True),
    )
    decoder = nn.Sequential(
        nn.Linear(saved["hidden_dim"], saved["output_dim"]), nn.ReLU(inplace=True)
    )
    model = nn.Sequential(
        encoder, decoder, nn.Linear(saved["output_dim"], saved["num_classes"])
    )
    model = model.to(device)
    model.load_state_dict(saved["model"])
    return model.eval(), np.asarray(saved["class_codes"], dtype="uint8")

## Load the checkpoint

Inference uses plain PyTorch. Change the device or batch size for your hardware.

In [2]:
source = Path("data/CGLC-MODIS-LCZ-100m.cog.tif")
checkpoint = Path("outputs/model.pt")
output_dir = Path("outputs/cogs")
device = "cpu"
batch_size = 4096
write_global_cogs = False
torch.set_num_threads(4)
model, class_codes = load_model(checkpoint, device)
print(model)

Sequential(
  (0): Sequential(
    (0): Wrap()
    (1): Linear(in_features=4, out_features=256, bias=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU(inplace=True)
  )
  (1): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU(inplace=True)
  )
  (2): Linear(in_features=256, out_features=31, bias=True)
)


## Write COGs

The default export covers the entire source grid. The 128-band embedding raster contains about 31.5 TB before compression, plus space for temporary rasters and overviews. Choose an output disk and set `write_global_cogs = True` to run. An optional `Window(col, row, width, height)` lets you check a subset.

Embedding overviews use averages; class overviews use modes. The source grid is nominally 100 m in EPSG:4326, so ground spacing varies with latitude.

In [3]:
@torch.inference_mode()
def export_cogs(model, class_codes, source, output_dir, window=None, batch_size=4096):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()
    encoder, head = model[0], model[1:]
    bands = model[1][0].in_features
    class_codes = np.asarray(class_codes, dtype="uint8")
    if batch_size < 1 or len(class_codes) != model[-1].out_features:
        raise ValueError("Check the batch size and class-code mapping.")
    embedding_path = output_dir / "embeddings_100m.cog.tif"
    prediction_path = output_dir / "land_cover_100m.cog.tif"
    palette_path = Path(source).with_name("cmap_Qgis_CGLC_MOD_LCZ.txt")
    colors = {0: (0, 0, 0, 0)}
    if palette_path.exists():
        with palette_path.open() as stream:
            colors.update(
                {int(row[0]): tuple(map(int, row[1:5])) for row in csv.reader(stream)}
            )

    with rasterio.open(source) as src, TemporaryDirectory(dir=output_dir) as temporary:
        if src.crs.to_epsg() != 4326:
            raise ValueError(
                "Expected the source longitude/latitude grid in EPSG:4326."
            )
        window = window or Window(0, 0, src.width, src.height)
        if (
            any(v != int(v) for v in window.flatten())
            or min(window.col_off, window.row_off) < 0
            or min(window.width, window.height) <= 0
            or window.col_off + window.width > src.width
            or window.row_off + window.height > src.height
        ):
            raise ValueError(
                "The export window must contain whole pixels within the source."
            )
        profile = {
            "driver": "GTiff",
            "crs": src.crs,
            "transform": src.window_transform(window),
            "width": int(window.width),
            "height": int(window.height),
            "tiled": True,
            "blockxsize": 512,
            "blockysize": 512,
            "compress": "zstd",
            "zstd_level": 3,
            "BIGTIFF": "YES",
            "NUM_THREADS": "4",
        }
        staged_embeddings = Path(temporary) / "embeddings.tif"
        staged_predictions = Path(temporary) / "predictions.tif"
        with (
            rasterio.open(
                staged_embeddings,
                "w",
                **profile,
                count=bands,
                dtype="float32",
                nodata=np.nan,
                predictor=3,
                interleave="band",
            ) as embeddings,
            rasterio.open(
                staged_predictions,
                "w",
                **profile,
                count=1,
                dtype="uint8",
                nodata=0,
                predictor=2,
            ) as predictions,
        ):
            if len(colors) > 1:
                predictions.write_colormap(1, colors)
            predictions.set_band_description(1, "Land cover class ID")
            for band in range(bands):
                embeddings.set_band_description(band + 1, f"embedding_{band}")
            for index, (_, tile) in enumerate(embeddings.block_windows(1), 1):
                source_tile = Window(
                    window.col_off + tile.col_off,
                    window.row_off + tile.row_off,
                    tile.width,
                    tile.height,
                )
                labels = src.read(1, window=source_tile, masked=True)
                valid = np.isin(labels.data, class_codes) & ~np.ma.getmaskarray(labels)
                rows, cols = np.indices(labels.shape)
                lon, lat = src.transform * (
                    cols + source_tile.col_off + 0.5,
                    rows + source_tile.row_off + 0.5,
                )
                coords = np.column_stack((lon.ravel(), lat.ravel())).astype("float32")
                features = np.full((labels.size, bands), np.nan, dtype="float32")
                classes = np.zeros(labels.size, dtype="uint8")
                ids = np.flatnonzero(valid.ravel())
                for start in range(0, len(ids), batch_size):
                    selected = ids[start : start + batch_size]
                    inputs = torch.from_numpy(coords[selected]).to(device)
                    z = encoder(inputs)
                    features[selected] = z.float().cpu().numpy()
                    classes[selected] = class_codes[head(z).argmax(1).cpu().numpy()]
                embeddings.write(features.T.reshape(bands, *labels.shape), window=tile)
                predictions.write(classes.reshape(labels.shape), 1, window=tile)
                if index % 100 == 0:
                    print(f"{index:,} tiles written", flush=True)
        for staged, final, resampling, predictor in (
            (staged_embeddings, embedding_path, "AVERAGE", "FLOATING_POINT"),
            (staged_predictions, prediction_path, "MODE", "STANDARD"),
        ):
            raster_copy(
                staged,
                final,
                driver="COG",
                COMPRESS="ZSTD",
                LEVEL=3,
                BLOCKSIZE=512,
                BIGTIFF="YES",
                NUM_THREADS="4",
                PREDICTOR=predictor,
                OVERVIEW_RESAMPLING=resampling,
            )
    return embedding_path, prediction_path

In [4]:
if write_global_cogs:
    embedding_path, prediction_path = export_cogs(
        model, class_codes, source, output_dir, batch_size=batch_size
    )
    print(embedding_path, prediction_path)

## Read embeddings

The bands are the middle bottleneck features, before expansion. Apply the saved linear probe directly to them.

In [5]:
if write_global_cogs:
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    head = nn.Linear(saved["hidden_dim"], saved["num_classes"])
    head.load_state_dict(saved["linear_head"])
    with rasterio.open(embedding_path) as src:
        tile = src.read(window=Window(0, 0, 256, 256))
    features = torch.from_numpy(tile.reshape(saved["hidden_dim"], -1).T.copy())
    valid = torch.isfinite(features).all(1)
    prediction = np.zeros(len(features), dtype="uint8")
    with torch.inference_mode():
        prediction[valid.numpy()] = class_codes[head(features[valid]).argmax(1).numpy()]
    prediction = prediction.reshape(tile.shape[1:])